In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm

csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
df['Delivery_Time'].hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution ({'Delivery_Time'})")
plt.xlabel('Delivery_Time')
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns ='Order_ID', axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
missing_values = df.isnull().sum()
print(missing_values) # Missing Values are relatively small; so I will just drop them.
df = df.dropna()

In [ ]:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()
print(duplicates)
df.drop_duplicates(inplace=True)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder

# Separate the target before scaling so it is not affected
X = df.drop("Delivery_Time", axis=1)
y = df['Delivery_Time'].astype(float)

categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))
for column in categorical_cols:
  print(pd.unique(df[column])) # Unique values in each categorical column

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  X[col] = le.fit_transform(X[col])
  label_encoders[col] = le

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_X = scaler.fit_transform(X)

In [ ]:
# Task 6: Write your code here:
## From the target distibution done in Part 1: Task 6, the target seems to be balanced somehow, as its distribution is close to the normal distribution.

In [ ]:
df.info()

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    scaled_X, y, test_size=0.2, random_state=42
)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{5}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  rf.fit(X_train, y_train)

  # Predict
  y_pred = rf.predict(X_test)

  # Calculate metrics
  mae_scores.append(mean_absolute_error(y_test, y_pred))

print(f"5-Fold CV Results:")
print(f"MAE:  ${np.mean(mae_scores):,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
predicted_delivery_time = pd.DataFrame(y_pred)
predicted_delivery_time.hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution ({'Predicted Delivery Time'})")
plt.xlabel('Predicted Delivery Time')
plt.ylabel("Frequency")
plt.grid(False)

plt.show()


In [ ]:
# Task Bonus: Write your code here: